# 02 · EDA + Preprocesamiento

**Proyecto:** ChurnLens · Diplomado MLDS — Universidad Nacional de Colombia
**Fase TDSP:** 2 · *Preprocesamiento y análisis exploratorio* (10 %)

**Objetivo de este notebook:**

1. Documentar el **análisis exploratorio reproducible** usando el subpaquete `churnlens.eda`.
2. Mostrar la **ingeniería de variables** derivadas que entran al pipeline (`churnlens.features.engineering`).
3. Ejecutar el pipeline completo de **preprocesamiento + split** (`churnlens.features.pipeline`).
4. Validar que la **partición preserva la tasa de churn** y dejar trazabilidad de los artefactos producidos.

> Todas las funciones usadas viven en el paquete instalable — este notebook es solo narrativa y verificación.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from churnlens.data.loader import TelcoChurnLoader
from churnlens.eda.plots import (
    plot_categorical_churn_rate,
    plot_correlation_heatmap,
    plot_missing_bar,
    plot_numeric_boxplot_by_target,
    plot_numeric_histogram,
    plot_target_distribution,
)
from churnlens.eda.report import generate_eda_report
from churnlens.eda.summary import (
    categorical_summary,
    churn_rate_by_category,
    cramers_v_vs_target,
    numeric_correlation,
    numeric_summary,
    target_distribution,
)
from churnlens.features.engineering import add_engineered_features
from churnlens.features.pipeline import run_preprocessing
from churnlens.features.preprocessing import (
    BINARY_CATEGORICAL_COLS,
    NOMINAL_CATEGORICAL_COLS,
    ORDINAL_COLS,
)
from churnlens.features.splits import stratified_split

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

print("Python:", sys.version.split()[0])
print("Project root:", PROJECT_ROOT)

## 1. Carga validada + features derivadas

Reutilizamos el _loader_ oficial de la Fase 1 y enriquecemos con las 7 features derivadas definidas en `churnlens.features.engineering.add_engineered_features`.

In [ ]:
loader = TelcoChurnLoader()
loader.download()
df_raw = loader.load_validated()
df = add_engineered_features(df_raw)

print(f"Filas crudas      : {len(df_raw):,}")
print(f"Cols crudas       : {df_raw.shape[1]}")
print(f"Cols con derivadas: {df.shape[1]}")
print(f"Nuevas columnas   : {sorted(set(df.columns) - set(df_raw.columns))}")

In [ ]:
df.head()

## 2. Distribución del target y valores faltantes

In [ ]:
target_distribution(df)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8))
plot_target_distribution(df, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
plot_missing_bar(df, ax=ax)
plt.tight_layout(); plt.show()

## 3. Estadísticas descriptivas

### 3.1 Numéricas

In [ ]:
numeric_summary(df, columns=["tenure", "MonthlyCharges", "TotalCharges",
                              "services_count", "avg_monthly_spend", "monthly_spend_gap"]).round(2)

### 3.2 Categóricas

In [ ]:
categorical_summary(df, columns=[*BINARY_CATEGORICAL_COLS, *ORDINAL_COLS, *NOMINAL_CATEGORICAL_COLS, "Churn"]).head(20)

## 4. Univariado numérico

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_numeric_histogram(df, "tenure", ax=axes[0])
plot_numeric_histogram(df, "MonthlyCharges", ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_numeric_boxplot_by_target(df, "TotalCharges", ax=ax)
plt.tight_layout(); plt.show()

**Observaciones:**

- `tenure` es claramente bimodal: gran masa en clientes nuevos (0-12 meses) y en clientes consolidados (49-72 meses).
- `MonthlyCharges` cubre el rango USD 18-119 sin sesgo extremo.
- `TotalCharges` por target: la mediana es **mucho menor** entre los _churners_, consistente con su menor antigüedad.

## 5. Bivariado — churn rate por categórica

In [ ]:
for col in ["Contract", "tenure_bucket", "PaymentMethod", "InternetService"]:
    print(f"\n=== Churn rate por {col} ===")
    print(churn_rate_by_category(df, col).round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.flatten(), ["Contract", "tenure_bucket", "PaymentMethod", "InternetService"]):
    plot_categorical_churn_rate(df, col, ax=ax)
plt.tight_layout(); plt.show()

**Hallazgos:**

- `Contract`: contratos `Month-to-month` churnean **15×** más que `Two year` (42.7 % vs 2.8 %).
- `tenure_bucket`: el bucket 0-12m churnea **5×** más que el 49-72m.
- `PaymentMethod`: `Electronic check` casi triplica la tasa de los métodos automáticos.
- `InternetService = Fiber optic` muestra churn anómalamente alto (41.9 %) — vale la pena indagar en Fase 3 si es problema de calidad de servicio o efecto de precio.

## 6. V de Cramér — fuerza de asociación con el target

In [ ]:
cramers = cramers_v_vs_target(
    df,
    [*BINARY_CATEGORICAL_COLS, *ORDINAL_COLS, *NOMINAL_CATEGORICAL_COLS],
)
cramers.round(3)

**Lectura:**

- Predictores fuertes (V > 0.30): `Contract`, `tenure_bucket`, `OnlineSecurity`, `TechSupport`, `InternetService`, `PaymentMethod`.
- Predictores nulos (V ≈ 0): `gender`, `PhoneService` — candidatos a descartar tras selección cuantitativa en Fase 3.

## 7. Correlaciones numéricas (Spearman)

In [ ]:
corr = numeric_correlation(
    df,
    columns=["tenure", "MonthlyCharges", "TotalCharges", "services_count", "avg_monthly_spend", "monthly_spend_gap"],
)
fig, ax = plt.subplots(figsize=(7, 5.5))
plot_correlation_heatmap(corr, ax=ax)
plt.tight_layout(); plt.show()
corr.round(2)

## 8. Pipeline de preprocesamiento

Ejecutamos el pipeline reproducible que ajusta el `ColumnTransformer` solo sobre `train` y materializa los tres parquet bajo `data/processed/`.

In [ ]:
artifacts = run_preprocessing()
print(json.dumps(artifacts.to_dict(), indent=2))

In [ ]:
meta = json.loads(artifacts.metadata_path.read_text("utf-8"))
print(json.dumps({k: v for k, v in meta.items() if k != "feature_names"}, indent=2))

### 8.1 Verificación del split

Las tasas de positivos deben ser ~iguales en train/val/test (≈26.5 %).

In [ ]:
split = stratified_split(df.drop(columns=["customerID"]))
print(json.dumps({"shapes": split.shapes, "target_rates": {k: round(v, 4) for k, v in split.target_rates.items()}}, indent=2))

## 9. Regeneración del reporte EDA persistente

Reproduce las 9 figuras y 4 tablas oficiales bajo `reports/`.

In [ ]:
report = generate_eda_report()
print("Figuras:")
for name, path in report.figures.items():
    print(f"  {name:30s} -> {path.name}")
print("\nTablas:")
for name, path in report.tables.items():
    print(f"  {name:30s} -> {path.name}")

## 10. Cierre

Este notebook valida que:

* El subpaquete `churnlens.eda` reproduce **todas** las estadísticas y figuras del reporte oficial.
* El subpaquete `churnlens.features` produce un dataset **listo para modelar** con 35 features tras el `ColumnTransformer`.
* El **split estratificado** preserva la tasa de churn dentro de ±0.1 pp.
* No hay valores faltantes después del preprocesamiento — la imputación ocurre dentro del transformer.

Próximo paso → **Fase 3 · Modelado.**

**Referencias:**
* [`docs/data/data_summary_report.md`](../docs/data/data_summary_report.md) — reporte oficial Fase 2.
* [`docs/data/data_dictionary.md`](../docs/data/data_dictionary.md) §6-7 — features derivadas y artefactos.
* [`docs/data/data_definition.md`](../docs/data/data_definition.md) §3.3, §5.4 — pipeline.